# LeetCode 1201 — Ugly Number III bằng Python

Link đề: https://leetcode.com/problems/ugly-number-iii/ · Độ khó: **Medium** · Chủ đề: Math, Binary Search, Combinatorics, Number Theory, Least Common Multiple, Euclidean Algorithm, Greatest Common Divisor, Inclusion-Exclusion Principle

Skeleton LeetCode:

```python
class Solution:
    def nthUglyNumber(self, n: int, a: int, b: int, c: int) -> int:
        ...
```

Ghi chú: đáp án có thể tới `2·10^9` nên **không thể liệt kê**. Hai công cụ then chốt: (1) **đếm** số "số xấu" `≤ x` bằng **nguyên lý bù trừ (inclusion–exclusion)** với **BCNN (LCM)**; (2) **tìm kiếm nhị phân trên đáp án** — tìm `x` nhỏ nhất có `count(x) ≥ n`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: "số xấu" ở đây là số nguyên dương **chia hết cho `a`, `b` hoặc `c`** (khác bài 263/264!), tìm số thứ `n`.
2. Tính **UCLN (gcd)** bằng thuật toán Euclid và **BCNN (lcm)** `= a·b / gcd(a, b)`.
3. Đếm số bội của `a` hoặc `b` hoặc `c` trong `[1, x]` bằng **bù trừ**: `x//a + x//b + x//c − x//lcm(a,b) − x//lcm(a,c) − x//lcm(b,c) + x//lcm(a,b,c)`.
4. Áp dụng **binary search trên đáp án** với hàm đơn điệu, nắm mẫu "tìm `x` nhỏ nhất thỏa điều kiện".
5. Tránh bẫy: dùng `a*b` thay cho `lcm`, tìm kiếm nhị phân trả về `x` không phải số xấu, vòng lặp nhị phân không dừng.

## 2. Đề bài & ví dụ

Một **số xấu (ugly number)** là số nguyên dương chia hết cho `a`, **hoặc** `b`, **hoặc** `c`. Cho bốn số nguyên `n, a, b, c`, trả về số xấu **thứ `n`** (đánh số từ 1, theo thứ tự tăng dần).

**Ví dụ 1:** `n = 3, a = 2, b = 3, c = 5` → `4`. Dãy: `2, 3, 4, 5, 6, 8, 9, 10, ...`

**Ví dụ 2:** `n = 4, a = 2, b = 3, c = 4` → `6`. Dãy: `2, 3, 4, 6, 8, 9, 10, 12, ...`

**Ví dụ 3:** `n = 5, a = 2, b = 11, c = 13` → `10`. Dãy: `2, 4, 6, 8, 10, 11, 12, 13, ...`

**Ràng buộc:** `1 <= n, a, b, c <= 10^9`; `1 <= a·b·c <= 10^18`; đáp án đảm bảo nằm trong `[1, 2·10^9]`.

In [1]:
import random
import time
from math import gcd

examples = [
    ((3, 2, 3, 5), 4),
    ((4, 2, 3, 4), 6),
    ((5, 2, 11, 13), 10),
    ((1000000000, 2, 217983653, 336916467), 1999999984),
]
for args, want in examples:
    print(args, "->", want)

(3, 2, 3, 5) -> 4
(4, 2, 3, 4) -> 6
(5, 2, 11, 13) -> 10
(1000000000, 2, 217983653, 336916467) -> 1999999984


## 3. Kiến thức nền

### 3.1 Phép chia lấy phần nguyên `//` để đếm bội số

Số bội của `a` trong đoạn `[1, x]` là `x // a`. Ví dụ bội của 3 trong `[1, 10]`: `3, 6, 9` → `10 // 3 = 3`.

In [2]:
x, a = 10, 3
print(x // a, [k for k in range(1, x + 1) if k % a == 0])

3 [3, 6, 9]


### 3.2 UCLN (gcd) — thuật toán Euclid

`gcd(a, b) = gcd(b, a % b)`, dừng khi `b = 0`. Python có sẵn `math.gcd`. Độ phức tạp `O(log min(a, b))`.

In [3]:
def my_gcd(a, b):
    while b:
        a, b = b, a % b
    return a


print(my_gcd(12, 18), gcd(12, 18), my_gcd(17, 5))
assert all(my_gcd(p, q) == gcd(p, q) for p in range(1, 40) for q in range(1, 40))

6 6 1


### 3.3 BCNN (lcm)

`lcm(a, b) = a // gcd(a, b) * b` (chia trước để số trung gian nhỏ hơn). Một số chia hết cho **cả** `a` và `b` ⇔ chia hết cho `lcm(a, b)`. Ba số: `lcm(a, b, c) = lcm(lcm(a, b), c)`.

**Chú ý:** `lcm(2, 4) = 4`, **không phải** `2·4 = 8`.

In [4]:
def lcm(a, b):
    return a // gcd(a, b) * b


print(lcm(2, 4), lcm(4, 6), lcm(lcm(2, 3), 5))
print("bội chung của 2 và 4 trong [1,20]:", [k for k in range(1, 21) if k % 2 == 0 and k % 4 == 0])

4 12 30
bội chung của 2 và 4 trong [1,20]: [4, 8, 12, 16, 20]


### 3.4 Nguyên lý bù trừ (inclusion–exclusion)

Với ba tập `A, B, C`:

`|A ∪ B ∪ C| = |A| + |B| + |C| − |A∩B| − |A∩C| − |B∩C| + |A∩B∩C|`.

Ở đây `A` = bội của `a` trong `[1, x]`, `|A| = x // a`, `|A ∩ B| = x // lcm(a, b)`, ...

In [5]:
def count_ugly(x, a, b, c):
    ab, ac, bc = lcm(a, b), lcm(a, c), lcm(b, c)
    abc = lcm(ab, c)
    return x // a + x // b + x // c - x // ab - x // ac - x // bc + x // abc


x, a, b, c = 30, 2, 3, 5
print(count_ugly(x, a, b, c), sum(1 for k in range(1, x + 1) if k % a == 0 or k % b == 0 or k % c == 0))

22 22


### 3.5 Tìm kiếm nhị phân trên đáp án

Nếu hàm `ok(x)` **đơn điệu** (sai, sai, ..., sai, đúng, đúng, ...), ta tìm `x` nhỏ nhất có `ok(x)` đúng trong `O(log(phạm vi))` lần gọi:

```python
lo, hi = L, R          # biết chắc ok(R) đúng
while lo < hi:
    mid = (lo + hi) // 2
    if ok(mid): hi = mid
    else:       lo = mid + 1
return lo
```

In [6]:
def first_true(lo, hi, ok):
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


print(first_true(0, 100, lambda v: v * v >= 50))   # số nhỏ nhất có bình phương >= 50

8


## 4. Ý tưởng nền tảng / trực giác

Với `a, b, c = 2, 3, 5`, lập bảng `count(x)` = số số xấu `≤ x`:

| x | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| số xấu? | | ✓ | ✓ | ✓ | ✓ | ✓ | | ✓ | ✓ | ✓ |
| count(x) | 0 | 1 | 2 | 3 | 4 | 5 | 5 | 6 | 7 | 8 |

`count` **không giảm**. Số xấu thứ `n = 3` là `x` nhỏ nhất có `count(x) ≥ 3` → `x = 4`. Chú ý `count(7) = count(6) = 5`: những `x` không phải số xấu không làm `count` tăng, nên "nhỏ nhất" tự động rơi vào số xấu.

In [7]:
a, b, c = 2, 3, 5
print([(x, count_ugly(x, a, b, c)) for x in range(1, 11)])

[(1, 0), (2, 1), (3, 2), (4, 3), (5, 4), (6, 5), (7, 5), (8, 6), (9, 7), (10, 8)]


## 5. Cách 1 — Brute Force / baseline

Duyệt `x = 1, 2, 3, ...`, đếm số chia hết cho `a`, `b` hoặc `c`; đến số thứ `n` thì trả về.

In [8]:
def brute(n: int, a: int, b: int, c: int) -> int:
    x = 0
    while n:
        x += 1
        if x % a == 0 or x % b == 0 or x % c == 0:
            n -= 1
    return x


for args, want in examples[:3]:
    print(brute(*args), want)
    assert brute(*args) == want

4 4
6 6
10 10


**Độ phức tạp:** `O(đáp án)` — tới `2·10^9` vòng lặp → hàng chục phút trong Python. **TLE**.

**Nút thắt:** ta kiểm tra **từng** số, kể cả những số không phải số xấu. Trong khi đó **đếm** được số số xấu `≤ x` trong `O(1)`.

In [9]:
for n in [10**4, 10**5, 10**6]:
    t = time.perf_counter(); brute(n, 7, 11, 13); print(n, f"{time.perf_counter() - t:.3f}s")

10000 0.003s
100000 0.029s


1000000 0.295s


## 6. Bước đột phá

**Insight:** "phần tử thứ `n` của một dãy tăng" ⇔ "`x` nhỏ nhất sao cho **có ít nhất `n` phần tử `≤ x`**". Nếu ta **đếm** được `count(x)` nhanh và `count` đơn điệu thì tìm kiếm nhị phân trên `x` là xong.

- `count(x)` tính bằng bù trừ trong `O(1)` (sau khi có sẵn các `lcm`).
- **Điều kiện:** `ok(x) = count(x) >= n` — đơn điệu theo `x`.
- **Phạm vi:** `lo = 1`, `hi = 2·10^9` (đề bảo đảm); chặt hơn: `hi = n · min(a, b, c)` (vì `n` bội đầu tiên của số nhỏ nhất đã là `n` số xấu).

**Invariant nhị phân:** đáp án luôn nằm trong `[lo, hi]`; `ok(hi)` luôn đúng.

## 7. Các cách tiếp theo

### Cách 2 — Trộn ba dãy bội (như Ugly Number II)

Ba "con trỏ" `ia, ib, ic` sinh bội tiếp theo `a·ia`, `b·ib`, `c·ic`; mỗi bước lấy min, tăng **mọi** con trỏ có giá trị bằng min (khử trùng). `O(n)` — tốt hơn brute (không duyệt số không xấu) nhưng `n = 10^9` vẫn quá chậm.

In [10]:
def solve_merge(n: int, a: int, b: int, c: int) -> int:
    ia = ib = ic = 1
    val = 0
    for _ in range(n):
        val = min(a * ia, b * ib, c * ic)
        if a * ia == val:
            ia += 1
        if b * ib == val:
            ib += 1
        if c * ic == val:
            ic += 1
    return val


for args, want in examples[:3]:
    assert solve_merge(*args) == want
print([solve_merge(*args) for args, _ in examples[:3]])

[4, 6, 10]


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### Cách 3 — Tìm kiếm nhị phân + bù trừ (lời giải chuẩn)

In [11]:
def solve_binary(n: int, a: int, b: int, c: int) -> int:
    ab, ac, bc = lcm(a, b), lcm(a, c), lcm(b, c)
    abc = lcm(ab, c)

    def count(x):
        return x // a + x // b + x // c - x // ab - x // ac - x // bc + x // abc

    lo, hi = 1, n * min(a, b, c)
    while lo < hi:
        mid = (lo + hi) // 2
        if count(mid) >= n:
            hi = mid
        else:
            lo = mid + 1
    return lo


for args, want in examples:
    assert solve_binary(*args) == want
print([solve_binary(*args) for args, _ in examples])

[4, 6, 10, 1999999984]


**Độ phức tạp:** `O(log(n · min(a,b,c)))` ≈ `O(log 2·10^9) ≈ 31` lần gọi `count`, mỗi lần `O(1)`; cộng `O(log)` cho các `gcd`. Bộ nhớ `O(1)`.

### Cách 4 — Tính chu kỳ + nhị phân trong một chu kỳ

Dãy số xấu **tuần hoàn** với chu kỳ `L = lcm(a, b, c)`: trong mỗi đoạn `[kL + 1, (k+1)L]` có đúng `P = count(L)` số xấu, và số `x` xấu ⇔ `x + L` xấu. Vậy:
- `q, r = divmod(n − 1, P)` → số thứ `n` = `q·L` + (số xấu thứ `r + 1` trong `[1, L]`).
- Phần dư tìm bằng nhị phân trên `[1, L]`.

Không nhanh hơn Cách 3 về bậc, nhưng minh họa cấu trúc tuần hoàn — hữu ích khi `n` lớn tới `10^18`.

In [12]:
def solve_period(n: int, a: int, b: int, c: int) -> int:
    ab, ac, bc = lcm(a, b), lcm(a, c), lcm(b, c)
    L = lcm(ab, c)

    def count(x):
        return x // a + x // b + x // c - x // ab - x // ac - x // bc + x // L

    P = count(L)
    q, r = divmod(n - 1, P)
    target = r + 1
    lo, hi = 1, L
    while lo < hi:
        mid = (lo + hi) // 2
        if count(mid) >= target:
            hi = mid
        else:
            lo = mid + 1
    return q * L + lo


for args, want in examples:
    assert solve_period(*args) == want
print([solve_period(*args) for args, _ in examples], "| n = 10^18:", solve_period(10**18, 2, 3, 5))

[4, 6, 10, 1999999984] | n = 10^18: 1363636363636363636


### Cách 5 — Bù trừ tổng quát cho `k` số (duyệt mọi tập con)

Với danh sách ước bất kỳ `divs`, `count(x) = Σ_{S ≠ ∅} (−1)^{|S|+1} · x // lcm(S)`. Cài bằng bitmask, dùng chung khung nhị phân. Hữu ích khi đề mở rộng sang 4, 5 số.

In [13]:
def count_general(x, divs):
    k = len(divs)
    total = 0
    for mask in range(1, 1 << k):
        l, bits = 1, 0
        for i in range(k):
            if mask >> i & 1:
                l = lcm(l, divs[i]); bits += 1
            if l > x:
                break
        total += (x // l) if bits % 2 == 1 else -(x // l)
    return total


def solve_general(n, divs):
    lo, hi = 1, n * min(divs)
    while lo < hi:
        mid = (lo + hi) // 2
        if count_general(mid, divs) >= n:
            hi = mid
        else:
            lo = mid + 1
    return lo


for (n, a, b, c), want in examples:
    assert solve_general(n, [a, b, c]) == want
print(solve_general(10, [2, 3, 5, 7]), brute(10, 2, 3, 5) if False else "")

12 


*(Lưu ý trong `count_general`: khi `l > x` ta `break` sớm vòng trong — nhưng khi đó `bits` chưa đếm hết; may mắn `x // l = 0` với mọi `l > x` nên đóng góp bằng 0 bất kể dấu, kết quả vẫn đúng.)*

## 8. TRACE từng bước

Trace Cách 3 với `n = 3, a = 2, b = 3, c = 5`: `lo = 1`, `hi = 3·2 = 6`.

| vòng | lo | hi | mid | count(mid) | ≥ 3? | cập nhật |
|---|---|---|---|---|---|---|
| 1 | 1 | 6 | 3 | 2 | không | lo = 4 |
| 2 | 4 | 6 | 5 | 4 | có | hi = 5 |
| 3 | 4 | 5 | 4 | 3 | có | hi = 4 |
| — | 4 | 4 | | | | dừng → **4** |

In [14]:
def trace_binary(n, a, b, c):
    ab, ac, bc = lcm(a, b), lcm(a, c), lcm(b, c)
    abc = lcm(ab, c)
    count = lambda x: x // a + x // b + x // c - x // ab - x // ac - x // bc + x // abc
    lo, hi = 1, n * min(a, b, c)
    print(f"lcm: ab={ab}, ac={ac}, bc={bc}, abc={abc}")
    step = 0
    while lo < hi:
        step += 1
        mid = (lo + hi) // 2
        cm = count(mid)
        if cm >= n:
            print(f"vòng {step}: lo={lo} hi={hi} mid={mid} count={cm} >= {n} -> hi={mid}")
            hi = mid
        else:
            print(f"vòng {step}: lo={lo} hi={hi} mid={mid} count={cm} <  {n} -> lo={mid + 1}")
            lo = mid + 1
    print("đáp án:", lo)
    return lo


trace_binary(3, 2, 3, 5)
print()
trace_binary(4, 2, 3, 4)

lcm: ab=6, ac=10, bc=15, abc=30
vòng 1: lo=1 hi=6 mid=3 count=2 <  3 -> lo=4
vòng 2: lo=4 hi=6 mid=5 count=4 >= 3 -> hi=5
vòng 3: lo=4 hi=5 mid=4 count=3 >= 3 -> hi=4
đáp án: 4

lcm: ab=6, ac=4, bc=12, abc=12
vòng 1: lo=1 hi=8 mid=4 count=3 <  4 -> lo=5
vòng 2: lo=5 hi=8 mid=6 count=4 >= 4 -> hi=6
vòng 3: lo=5 hi=6 mid=5 count=3 <  4 -> lo=6
đáp án: 6


6

In [15]:
# Số vòng lặp với input lớn nhất: chỉ ~31 vòng
def count_steps(n, a, b, c):
    lo, hi, steps = 1, n * min(a, b, c), 0
    ab, ac, bc = lcm(a, b), lcm(a, c), lcm(b, c); abc = lcm(ab, c)
    while lo < hi:
        steps += 1
        mid = (lo + hi) // 2
        if mid // a + mid // b + mid // c - mid // ab - mid // ac - mid // bc + mid // abc >= n:
            hi = mid
        else:
            lo = mid + 1
    return lo, steps


print(count_steps(10**9, 2, 217983653, 336916467))

(1999999984, 31)


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def nthUglyNumber(self, n: int, a: int, b: int, c: int) -> int:
        def lcm(x: int, y: int) -> int:
            return x // gcd(x, y) * y

        ab, ac, bc = lcm(a, b), lcm(a, c), lcm(b, c)
        abc = lcm(ab, c)

        def count(x: int) -> int:
            return x // a + x // b + x // c - x // ab - x // ac - x // bc + x // abc

        lo, hi = 1, 2 * 10**9
        while lo < hi:
            mid = (lo + hi) // 2
            if count(mid) >= n:
                hi = mid
            else:
                lo = mid + 1
        return lo


sol = Solution()
for args, want in examples:
    print(sol.nthUglyNumber(*args))
    assert sol.nthUglyNumber(*args) == want

4
6
10
1999999984


## 10. Vì sao đúng

1. **`count(x)` đúng:** theo bù trừ, mỗi số `k ≤ x` chia hết cho ít nhất một trong `a, b, c` được đếm đúng một lần (nếu chia hết cho đúng `t` số trong ba, nó được cộng `t`, trừ `C(t,2)`, cộng `C(t,3)` lần → `t − C(t,2) + C(t,3) = 1` với `t = 1, 2, 3`). Và "chia hết cho cả `a` và `b`" ⇔ "chia hết cho `lcm(a, b)`".
2. **Đơn điệu:** `count(x+1) − count(x) ∈ {0, 1}`.
3. **Nhị phân tìm `x*` nhỏ nhất có `count(x*) ≥ n`:** khi đó `count(x* − 1) < n ≤ count(x*)` nên `count(x*) = count(x* − 1) + 1` (bước nhảy tối đa 1) → `x*` **là số xấu** và có đúng `n − 1` số xấu nhỏ hơn nó → `x*` là số xấu thứ `n`.
4. **Phạm vi:** `count(n · min(a,b,c)) ≥ n` nên `hi` hợp lệ; đề cũng bảo đảm đáp án `≤ 2·10^9`.

In [17]:
# Kiểm chứng t - C(t,2) + C(t,3) = 1
from math import comb
print([t - comb(t, 2) + comb(t, 3) for t in (1, 2, 3)])

[1, 1, 1]


## 11. Time / Space Complexity

Gọi `R` = cận trên tìm kiếm (`≤ 2·10^9`).

| Cách | Time | Space |
|---|---|---|
| 1. Brute | `O(đáp án)` ≤ `O(2·10^9)` | `O(1)` |
| 2. Trộn ba dãy | `O(n)` | `O(1)` |
| 3. Nhị phân + bù trừ | `O(log R + log max(a,b,c))` | `O(1)` |
| 4. Chu kỳ + nhị phân | `O(log L)` | `O(1)` |
| 5. Bù trừ tổng quát `k` số | `O(2^k · k · log R)` | `O(1)` |

Python không lo tràn số: `lcm` có thể tới `10^18`, vẫn là số nguyên chính xác.

## 12. Bộ test

In [18]:
solvers = {
    "merge": solve_merge,
    "binary": solve_binary,
    "period": solve_period,
    "general": lambda n, a, b, c: solve_general(n, [a, b, c]),
    "submit": Solution().nthUglyNumber,
}
edge = [
    ((1, 1, 1, 1), 1),
    ((7, 1, 5, 9), 7),
    ((5, 2, 2, 2), 10),
    ((3, 4, 2, 8), 6),
    ((1, 7, 11, 13), 7),
    ((6, 6, 10, 15), 20),
]
for args, want in examples[:3] + edge:
    assert brute(*args) == want, (args, brute(*args))
    for name, f in solvers.items():
        assert f(*args) == want, (name, args, f(*args), want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [19]:
random.seed(1201)
for trial in range(1500):
    a, b, c = (random.randint(1, 30) for _ in range(3))
    n = random.randint(1, 200)
    want = brute(n, a, b, c)
    for name, f in solvers.items():
        assert f(n, a, b, c) == want, (name, n, a, b, c)
print("1500 test ngẫu nhiên khớp brute force")

1500 test ngẫu nhiên khớp brute force


In [20]:
# Test lớn: đối chiếu binary / period / submit với nhau (brute không chạy nổi)
random.seed(7)
t = time.perf_counter()
cnt = 0
while cnt < 2000:
    a, b, c = (random.randint(1, 10**9) for _ in range(3))
    if a * b * c > 10**18:
        continue
    n = random.randint(1, 10**9)
    if n * min(a, b, c) > 2 * 10**9:
        n = max(1, 2 * 10**9 // min(a, b, c))
    r1 = solve_binary(n, a, b, c)
    assert r1 == solve_period(n, a, b, c) == Solution().nthUglyNumber(n, a, b, c)
    cnt += 1
print(f"2000 test lớn khớp nhau, {time.perf_counter() - t:.2f}s")
assert Solution().nthUglyNumber(10**9, 2, 217983653, 336916467) == 1999999984
assert Solution().nthUglyNumber(10**9, 2, 3, 5) == solve_period(10**9, 2, 3, 5)

KeyboardInterrupt: 

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng tích `a*b` thay cho `lcm(a, b)`

In [ ]:
def wrong_product(n, a, b, c):
    count = lambda x: x // a + x // b + x // c - x // (a * b) - x // (a * c) - x // (b * c) + x // (a * b * c)
    lo, hi = 1, 2 * 10**9
    while lo < hi:
        mid = (lo + hi) // 2
        if count(mid) >= n:
            hi = mid
        else:
            lo = mid + 1
    return lo


print(wrong_product(4, 2, 3, 4), "đúng là", brute(4, 2, 3, 4))

**Vì sao sai:** `2` và `4` không nguyên tố cùng nhau; số chia hết cho cả `2` và `4` là bội của `4`, không phải bội của `8`. Đếm sai → `count` quá lớn → trả số quá nhỏ (ở đây `4` thay vì `6`). **Sửa:** dùng `lcm`.

### Lỗi 2 — Dừng nhị phân khi `count(mid) == n` và trả `mid`

In [ ]:
def wrong_equal_stop(n, a, b, c):
    ab, ac, bc = lcm(a, b), lcm(a, c), lcm(b, c); abc = lcm(ab, c)
    count = lambda x: x // a + x // b + x // c - x // ab - x // ac - x // bc + x // abc
    lo, hi = 1, 2 * 10**9
    while lo <= hi:
        mid = (lo + hi) // 2
        k = count(mid)
        if k == n:
            return mid                  # SAI: mid có thể không phải số xấu
        if k < n:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1


print(wrong_equal_stop(3, 2, 3, 5), wrong_equal_stop(5, 2, 11, 13), "| đúng là", brute(3, 2, 3, 5), brute(5, 2, 11, 13))

**Vì sao sai:** nhiều `x` liên tiếp có cùng `count(x) = n` (ví dụ `count(10) = count(10..) = 5` cho tới trước 11). `mid` rơi vào giữa đoạn đó có thể **không chia hết** cho `a, b, c`. Phải tìm `x` **nhỏ nhất** có `count ≥ n`. **Sửa:** mẫu `first_true`. (Nếu muốn giữ kiểu này thì phải lùi `mid` về số xấu gần nhất `≤ mid`: `max(mid//a*a, mid//b*b, mid//c*c)`.)

### Lỗi 3 — Cập nhật `hi = mid - 1` khi `count(mid) >= n` (mất đáp án)

In [ ]:
def wrong_hi(n, a, b, c):
    ab, ac, bc = lcm(a, b), lcm(a, c), lcm(b, c); abc = lcm(ab, c)
    count = lambda x: x // a + x // b + x // c - x // ab - x // ac - x // bc + x // abc
    lo, hi = 1, 2 * 10**9
    while lo < hi:
        mid = (lo + hi) // 2
        if count(mid) >= n:
            hi = mid - 1                # SAI: mid có thể chính là đáp án
        else:
            lo = mid + 1
    return lo


bad = [(args, wrong_hi(*args), brute(*args)) for args, _ in examples[:3] if wrong_hi(*args) != brute(*args)]
print(bad)

**Vì sao sai:** khi `count(mid) ≥ n`, `mid` vẫn có thể là đáp án; `hi = mid − 1` loại nó đi. **Sửa:** `hi = mid` (và vòng lặp `while lo < hi`).

### Lỗi 4 — Trộn ba dãy nhưng chỉ tăng **một** con trỏ khi trùng

In [ ]:
def wrong_merge(n, a, b, c):
    ia = ib = ic = 1
    val = 0
    for _ in range(n):
        val = min(a * ia, b * ib, c * ic)
        if a * ia == val:
            ia += 1
        elif b * ib == val:             # SAI: elif -> giá trị trùng bị đếm 2 lần
            ib += 1
        else:
            ic += 1
    return val


print(wrong_merge(4, 2, 3, 4), "đúng là", brute(4, 2, 3, 4))

**Vì sao sai:** `6` là bội của cả `2` và `3`; dùng `elif` chỉ tăng con trỏ của `2`, lần sau `6` lại xuất hiện từ dãy `3` → đếm trùng. **Sửa:** dùng ba `if` độc lập.

## 14. Mẹo & nhận dạng pattern

- **"Phần tử thứ `k`" của dãy tăng mà không liệt kê được** → binary search trên giá trị + hàm đếm `count(x)`.
- **"Chia hết cho a hoặc b hoặc c"** → bù trừ với `lcm`.
- **Mẫu `first_true`**: `while lo < hi: mid; if ok: hi = mid else lo = mid + 1`.
- Kiểm tra: đáp án của nhị phân có thực sự thuộc tập cần tìm không? (ở đây có, nhờ bước nhảy `count` ≤ 1).
- Phân biệt các bài "Ugly Number" (263, 264, 313, 1201) — định nghĩa khác nhau hoàn toàn.

## 15. Biến thể

1. **LeetCode 878 — Nth Magical Number:** chỉ hai số `a, b` → bù trừ 2 tập.
2. **LeetCode 264 — Ugly Number II:** số chỉ có ước nguyên tố 2, 3, 5 → ba con trỏ (khác định nghĩa!).
3. **LeetCode 668 / 719 — kth smallest trong bảng nhân / cặp khoảng cách:** cùng khung "nhị phân + đếm".
4. **`k` ước số bất kỳ** → Cách 5 (bitmask bù trừ).
5. **Đếm số trong `[L, R]` chia hết cho a/b/c** → `count(R) − count(L − 1)`.

In [ ]:
def nth_magical(n, a, b):
    """Biến thể 1 (LeetCode 878), modulo 1e9+7."""
    l = lcm(a, b)
    lo, hi = 1, n * min(a, b)
    while lo < hi:
        mid = (lo + hi) // 2
        if mid // a + mid // b - mid // l >= n:
            hi = mid
        else:
            lo = mid + 1
    return lo % (10**9 + 7)


def count_range(L, R, a, b, c):
    return count_ugly(R, a, b, c) - count_ugly(L - 1, a, b, c)


print(nth_magical(4, 2, 3), nth_magical(5, 2, 4), count_range(10, 20, 2, 3, 5))
assert count_range(10, 20, 2, 3, 5) == sum(1 for k in range(10, 21) if k % 2 == 0 or k % 3 == 0 or k % 5 == 0)

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Duyệt từng số | `O(ans)` | `O(1)` | Oracle với input nhỏ |
| 2. Trộn ba dãy | Ba con trỏ bội | `O(n)` | `O(1)` | `n` nhỏ, cần liệt kê cả dãy |
| 3. Nhị phân + bù trừ | `x` nhỏ nhất có `count ≥ n` | `O(log R)` | `O(1)` | Nộp bài |
| 4. Chu kỳ | `L = lcm(a,b,c)`, chia `n` theo chu kỳ | `O(log L)` | `O(1)` | `n` cực lớn |
| 5. Bù trừ tổng quát | Bitmask trên `k` ước | `O(2^k·k·log R)` | `O(1)` | Nhiều hơn 3 ước |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `nthUglyNumber(6, 2, 3, 5)` = ?

**Bài 2 (tính tay):** tính `count(20)` với `a, b, c = 4, 6, 10` bằng bù trừ.

**Bài 3 (điền TODO):** hoàn thiện `count3(x, a, b, c)`.

**Bài 4 (tự viết lại):** viết lại lời giải nhị phân với `lo = 0`, `hi` là số mà `count(hi) ≥ n`, theo kiểu "nửa mở" `(lo, hi]`.

**Bài 5 (mini challenge):** tìm số thứ `n` chia hết cho `a` hoặc `b` nhưng **không** chia hết cho `c`.

In [ ]:
def count3(x, a, b, c):
    # TODO: bù trừ 3 tập
    pass


def nth_half_open(n, a, b, c):
    # TODO: bất biến count(lo) < n <= count(hi)
    raise NotImplementedError


def nth_ab_not_c(n, a, b, c):
    # TODO
    raise NotImplementedError

# assert count3(20, 4, 6, 10) == 8
# assert nth_ab_not_c(3, 2, 3, 6) == 4

## 18. Gợi ý 3 mức

- **Mức 1:** Đừng liệt kê. Hỏi ngược: có bao nhiêu số xấu `≤ x`?
- **Mức 2:** `count(x)` dùng bù trừ với `lcm`; `count` đơn điệu → nhị phân.
- **Mức 3:** Bài 5: `count'(x) = |(A ∪ B) \ C| = |A ∪ B| − |(A ∪ B) ∩ C| = (x//a + x//b − x//lcm(a,b)) − (x//lcm(a,c) + x//lcm(b,c) − x//lcm(a,b,c))`. Cần cận trên đủ lớn (ví dụ `n · lcm(...)`) hoặc tăng gấp đôi `hi` đến khi `count' ≥ n`.

## 19. Đáp án tham khảo

**Bài 1:** dãy `2,3,4,5,6,8,...` → số thứ 6 là `8`.

**Bài 2:** `20//4 + 20//6 + 20//10 − 20//12 − 20//20 − 20//30 + 20//60 = 5 + 3 + 2 − 1 − 1 − 0 + 0 = 8`.

In [ ]:
assert Solution().nthUglyNumber(6, 2, 3, 5) == 8


def count3(x, a, b, c):
    ab, ac, bc = lcm(a, b), lcm(a, c), lcm(b, c)
    return x // a + x // b + x // c - x // ab - x // ac - x // bc + x // lcm(ab, c)


def nth_half_open(n, a, b, c):
    lo, hi = 0, n * min(a, b, c)        # count(0) = 0 < n <= count(hi)
    while hi - lo > 1:
        mid = (lo + hi) // 2
        if count3(mid, a, b, c) >= n:
            hi = mid
        else:
            lo = mid
    return hi


def nth_ab_not_c(n, a, b, c):
    ab, ac, bc = lcm(a, b), lcm(a, c), lcm(b, c)
    abc = lcm(ab, c)
    cnt = lambda x: (x // a + x // b - x // ab) - (x // ac + x // bc - x // abc)
    hi = 1
    while cnt(hi) < n:
        hi *= 2
        if hi > 10**15:
            return -1                   # không đủ số (ví dụ c chia hết cả a và b)
    lo = 0
    while hi - lo > 1:
        mid = (lo + hi) // 2
        if cnt(mid) >= n:
            hi = mid
        else:
            lo = mid
    return hi


def brute_ab_not_c(n, a, b, c):
    x = 0
    while n:
        x += 1
        if (x % a == 0 or x % b == 0) and x % c != 0:
            n -= 1
    return x


assert count3(20, 4, 6, 10) == 8
assert nth_ab_not_c(3, 2, 3, 6) == 4
random.seed(19)
for _ in range(500):
    a, b, c = (random.randint(1, 20) for _ in range(3))
    n = random.randint(1, 60)
    assert nth_half_open(n, a, b, c) == brute(n, a, b, c)
    if c % a != 0 or c % b != 0:      # còn vô hạn số thỏa
        assert nth_ab_not_c(n, a, b, c) == brute_ab_not_c(n, a, b, c), (n, a, b, c)
print("Đáp án bài tập: OK")

## 20. Template nhớ nhanh + Checklist trước khi submit

```python
lcm = lambda x, y: x // gcd(x, y) * y
ab, ac, bc = lcm(a, b), lcm(a, c), lcm(b, c); abc = lcm(ab, c)
count = lambda x: x//a + x//b + x//c - x//ab - x//ac - x//bc + x//abc
lo, hi = 1, 2 * 10**9
while lo < hi:
    mid = (lo + hi) // 2
    if count(mid) >= n: hi = mid
    else:               lo = mid + 1
return lo
```

**Checklist:**
- [ ] Dùng `lcm`, không dùng tích.
- [ ] `lcm` tính `x // gcd * y` (chia trước).
- [ ] Mẫu nhị phân "nhỏ nhất thỏa": `hi = mid`, `lo = mid + 1`, `while lo < hi`.
- [ ] Không dừng khi `count == n` (có thể không phải số xấu).
- [ ] `hi` đủ lớn (`2·10^9` hoặc `n·min(a,b,c)`).
- [ ] Test `a = b = c`, một trong ba bằng `1`, các số không nguyên tố cùng nhau.